# Qdrant RAG Operations

This notebook is a direct Qdrant operations and observability console for the RAG collections.
It intentionally does not use `rag.ops`, `rag.sources.cli`, or LlamaIndex.

Use it for:
- checking physical collections and alias mappings
- inspecting collection vector schema, status, and counts
- reading compact Qdrant-side collection attestations
- sampling payloads and diagnosing metadata shape
- finding stale, orphaned, or suspicious collections
- carefully repairing aliases or deleting collections after explicit confirmation

Use the CLI for lifecycle actions like build, bundle, materialize, and promote. Use LlamaIndex-facing code for retrieval behavior checks. This notebook stays close to Qdrant so the database state is visible without extra abstractions.

## 1. Connect

Run this first. It uses the same project settings as the services, so inside Docker it should resolve Qdrant by service name.

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
from typing import Any

repo_root = next(
    (
        candidate.resolve()
        for candidate in (
            Path.cwd(),
            Path.cwd().parent,
            Path.cwd().parent.parent,
            Path('/opt/airflow/project'),
            Path('/home/jovyan'),
        )
        if (candidate / 'src' / 'rag').exists()
    ),
    Path.cwd().resolve(),
)
for candidate in (repo_root / 'src', repo_root):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

from qdrant_client import QdrantClient
from qdrant_client.http import models as qm

from rag.sources.materialize import qdrant_alias_name
from rag.vector_store import QdrantVectorStore
from shared.catalog import build_kb_index, get_catalog
from shared.config import get_settings

settings = get_settings()
client = QdrantClient(host=settings.platform.qdrant_host, port=settings.platform.qdrant_port)

print(f'Repo root: {repo_root}')
print(f'Qdrant: {settings.platform.qdrant_host}:{settings.platform.qdrant_port}')
print(client.get_collections())

## 2. Helpers

Small formatting helpers used by the rest of the notebook. They avoid project-level RAG wrappers but keep Qdrant response objects readable.

In [ ]:
def as_dict(value: Any) -> Any:
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if hasattr(value, 'dict'):
        return value.dict()
    if isinstance(value, list):
        return [as_dict(item) for item in value]
    if isinstance(value, tuple):
        return tuple(as_dict(item) for item in value)
    if isinstance(value, dict):
        return {key: as_dict(item) for key, item in value.items()}
    return value


def collection_names() -> list[str]:
    return sorted(collection.name for collection in client.get_collections().collections)


def alias_rows() -> list[dict[str, str]]:
    return sorted(
        [
            {'alias_name': alias.alias_name, 'collection_name': alias.collection_name}
            for alias in client.get_aliases().aliases
        ],
        key=lambda row: (row['alias_name'], row['collection_name']),
    )


def resolve_alias(name: str) -> str | None:
    for row in alias_rows():
        if row['alias_name'] == name:
            return row['collection_name']
    return None


def target_collection(name: str) -> str:
    return resolve_alias(name) or name


def vector_store(name: str) -> QdrantVectorStore:
    return QdrantVectorStore(
        host=settings.platform.qdrant_host,
        port=settings.platform.qdrant_port,
        collection_name=target_collection(name),
    )


def vector_schema(info: Any) -> dict[str, Any]:
    params = getattr(getattr(getattr(info, 'config', None), 'params', None), 'vectors', None)
    sparse = getattr(getattr(getattr(info, 'config', None), 'params', None), 'sparse_vectors', None)
    return {'vectors': as_dict(params), 'sparse_vectors': as_dict(sparse)}


def collection_summary(name: str) -> dict[str, Any]:
    info = client.get_collection(name)
    return {
        'collection': name,
        'status': str(getattr(info, 'status', None)),
        'optimizer_status': str(getattr(info, 'optimizer_status', None)),
        'points_count': getattr(info, 'points_count', None),
        'indexed_vectors_count': getattr(info, 'indexed_vectors_count', None),
        'segments_count': getattr(info, 'segments_count', None),
        'vectors': vector_schema(info)['vectors'],
        'sparse_vectors': vector_schema(info)['sparse_vectors'],
    }


def print_json(value: Any) -> None:
    print(json.dumps(as_dict(value), indent=2, sort_keys=True, default=str))

## 3. Catalog Alias Expectations

This derives expected Qdrant alias names from the catalog. It is useful for spotting missing or extra aliases after a deploy or manual promotion.

In [ ]:
catalog = get_catalog()
expected_aliases = []
for kb_id, kb_cfg in sorted(catalog.knowledge_bases.items()):
    for alias in sorted(kb_cfg.aliases):
        expected_aliases.append(
            {
                'kb_id': kb_id,
                'alias': alias,
                'qdrant_alias': qdrant_alias_name(kb_id=kb_id, alias=alias),
                'retrieval_strategy': kb_cfg.aliases[alias].retrieval_strategy,
                'reranker': kb_cfg.aliases[alias].reranker,
            }
        )

pprint(expected_aliases)

## 4. Inventory: Collections And Aliases

Read-only overview of everything currently visible in Qdrant.

In [ ]:
collections = collection_names()
aliases = alias_rows()

print(f'Collections: {len(collections)}')
pprint(collections)
print(f'\nAliases: {len(aliases)}')
pprint(aliases)

## 5. Collection Summary Table

Shows counts, vector legs, sparse-vector presence, and Qdrant status for every physical collection.

In [ ]:
summaries = []
for name in collection_names():
    try:
        summaries.append(collection_summary(name))
    except Exception as exc:
        summaries.append({'collection': name, 'error': f'{type(exc).__name__}: {exc}'})

pprint(summaries)

## 6. Inspect One Collection Or Alias

Set `TARGET_NAME` to either a physical collection or an alias like `rag__pytorch_reference__champion`.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'

resolved = resolve_alias(TARGET_NAME)
collection = target_collection(TARGET_NAME)
print_json(
    {
        'target_name': TARGET_NAME,
        'resolved_collection': resolved,
        'summary': collection_summary(collection),
        'raw_collection_info': as_dict(client.get_collection(collection)),
    }
)

## 7. Read Collection Attestation

Materialized collections write a compact metadata point into Qdrant. This is not the full build manifest; it is the runtime attestation used to validate alias targets.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'

store = vector_store(TARGET_NAME)
payload = store.read_meta()
print_json(
    {
        'target_name': TARGET_NAME,
        'resolved_collection': target_collection(TARGET_NAME),
        'attestation_payload': payload,
    }
)

## 8. Alias Health Check

Compares catalog-expected aliases with live Qdrant aliases and checks whether live aliases point to existing collections.

In [ ]:
live_aliases = {row['alias_name']: row['collection_name'] for row in alias_rows()}
live_collections = set(collection_names())
expected_alias_names = {row['qdrant_alias'] for row in expected_aliases}

health = {
    'expected_missing': sorted(expected_alias_names - set(live_aliases)),
    'unexpected_rag_aliases': sorted(
        alias for alias in live_aliases if alias.startswith('rag__') and alias not in expected_alias_names
    ),
    'dangling_aliases': sorted(
        (
            {'alias_name': alias, 'collection_name': collection}
            for alias, collection in live_aliases.items()
            if collection not in live_collections
        ),
        key=lambda row: row['alias_name'],
    ),
    'live_expected_aliases': sorted(
        (
            {'alias_name': alias, 'collection_name': collection}
            for alias, collection in live_aliases.items()
            if alias in expected_alias_names
        ),
        key=lambda row: row['alias_name'],
    ),
}

print_json(health)

## 9. Orphans And Stale Candidates

A collection is "orphaned" here if no alias points to it. That does not automatically mean it should be deleted: recent challenger builds, failed experiments, and rollback candidates can all be intentionally orphaned.

In [ ]:
aliased_collections = {row['collection_name'] for row in alias_rows()}
all_collections = set(collection_names())
orphan_collections = sorted(all_collections - aliased_collections)

orphan_summaries = []
for name in orphan_collections:
    row = collection_summary(name)
    row['attestation'] = vector_store(name).read_meta()
    orphan_summaries.append(row)

print_json(
    {
        'orphan_collection_count': len(orphan_collections),
        'orphan_collections': orphan_summaries,
    }
)

## 10. Sample Points

Scroll a few points from a collection. Metadata sentinel points are excluded by default.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'
LIMIT = 5
INCLUDE_VECTORS = False
EXCLUDE_ATTESTATION = True

collection = target_collection(TARGET_NAME)
scroll_filter = None
if EXCLUDE_ATTESTATION:
    scroll_filter = qm.Filter(
        must_not=[
            qm.FieldCondition(
                key='type',
                match=qm.MatchValue(value='collection_meta'),
            )
        ]
    )

points, next_page = client.scroll(
    collection_name=collection,
    scroll_filter=scroll_filter,
    limit=LIMIT,
    with_payload=True,
    with_vectors=INCLUDE_VECTORS,
)

print_json(
    {
        'target_name': TARGET_NAME,
        'collection': collection,
        'returned': len(points),
        'next_page_offset': next_page,
        'points': [as_dict(point) for point in points],
    }
)

## 11. Payload Keys And Sample Values

Scans a bounded number of points and summarizes which payload keys exist. This helps catch malformed materialization payloads without dumping the whole collection.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'
MAX_POINTS = 500
PAGE_SIZE = 100

collection = target_collection(TARGET_NAME)
seen = 0
next_page = None
key_counts = Counter()
sample_values: dict[str, list[Any]] = defaultdict(list)

while seen < MAX_POINTS:
    points, next_page = client.scroll(
        collection_name=collection,
        offset=next_page,
        limit=min(PAGE_SIZE, MAX_POINTS - seen),
        with_payload=True,
        with_vectors=False,
        scroll_filter=qm.Filter(
            must_not=[qm.FieldCondition(key='type', match=qm.MatchValue(value='collection_meta'))]
        ),
    )
    if not points:
        break
    for point in points:
        payload = point.payload or {}
        for key, value in payload.items():
            key_counts[key] += 1
            if len(sample_values[key]) < 3:
                sample_values[key].append(value)
    seen += len(points)
    if next_page is None:
        break

print_json(
    {
        'collection': collection,
        'scanned_points': seen,
        'key_counts': dict(sorted(key_counts.items())),
        'sample_values': dict(sorted(sample_values.items())),
    }
)

## 12. Count Points By Payload Value

Set a payload key and values to get quick distribution checks. This uses exact-match filters, so it is best for fields like `source_type`, `source_document_id`, `document_id`, or `metadata_kind`.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'
PAYLOAD_KEY = 'source_type'
PAYLOAD_VALUES = ['docs', 'arxiv']

collection = target_collection(TARGET_NAME)
counts = {}
for value in PAYLOAD_VALUES:
    result = client.count(
        collection_name=collection,
        count_filter=qm.Filter(
            must=[qm.FieldCondition(key=PAYLOAD_KEY, match=qm.MatchValue(value=value))]
        ),
        exact=True,
    )
    counts[value] = result.count

print_json({'collection': collection, 'payload_key': PAYLOAD_KEY, 'counts': counts})

## 13. Find Chunks For One Source Document

Useful when a source looks missing or duplicated.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'
SOURCE_DOCUMENT_ID = 'torch.nn'
LIMIT = 20

collection = target_collection(TARGET_NAME)
points, next_page = client.scroll(
    collection_name=collection,
    limit=LIMIT,
    with_payload=True,
    with_vectors=False,
    scroll_filter=qm.Filter(
        must=[qm.FieldCondition(key='source_document_id', match=qm.MatchValue(value=SOURCE_DOCUMENT_ID))],
        must_not=[qm.FieldCondition(key='type', match=qm.MatchValue(value='collection_meta'))],
    ),
)

print_json(
    {
        'collection': collection,
        'source_document_id': SOURCE_DOCUMENT_ID,
        'returned': len(points),
        'next_page_offset': next_page,
        'chunks': [as_dict(point) for point in points],
    }
)

## 14. Raw Dense Vector Search

This is a Qdrant plumbing check, not a product retrieval test. It verifies that the dense vector leg can be queried when you provide an embedding vector.

In [ ]:
TARGET_NAME = 'rag__pytorch_reference__champion'
QUERY_VECTOR = None  # Paste a dense vector list here, or leave None to skip.
TOP_K = 5

if QUERY_VECTOR is None:
    print('Set QUERY_VECTOR to a dense vector list to run a raw Qdrant search.')
else:
    collection = target_collection(TARGET_NAME)
    result = client.query_points(
        collection_name=collection,
        query=QUERY_VECTOR,
        using='dense',
        limit=TOP_K,
        with_payload=True,
        query_filter=qm.Filter(
            must_not=[qm.FieldCondition(key='type', match=qm.MatchValue(value='collection_meta'))]
        ),
    )
    print_json(as_dict(result.points))

## 15. Create Snapshot

Manual backup operation for one collection. Useful before alias surgery or collection cleanup.

In [ ]:
SNAPSHOT_COLLECTION = None  # Example: 'rag__pytorch_reference__challenger__20260605_120000'
CONFIRM_CREATE_SNAPSHOT = False

if SNAPSHOT_COLLECTION and CONFIRM_CREATE_SNAPSHOT:
    result = client.create_snapshot(collection_name=SNAPSHOT_COLLECTION)
    print_json(result)
else:
    print('Set SNAPSHOT_COLLECTION and CONFIRM_CREATE_SNAPSHOT = True to create a snapshot.')

## 16. Alias Repair

Direct alias changes bypass lifecycle validation. Prefer `python -m rag.sources.cli promote-alias` for normal promotion. Use this only for manual repair after you have inspected the target collection and its attestation.

In [ ]:
ALIAS_ACTION = None  # 'point' or 'delete'
ALIAS_NAME = 'rag__pytorch_reference__challenger'
ALIAS_TARGET_COLLECTION = 'rag__pytorch_reference__challenger__YYYYMMDD_HHMMSS'
CONFIRM_ALIAS_ACTION = False

if ALIAS_ACTION == 'point' and CONFIRM_ALIAS_ACTION:
    client.update_collection_aliases(
        change_aliases_operations=[
            qm.CreateAliasOperation(
                create_alias=qm.CreateAlias(
                    collection_name=ALIAS_TARGET_COLLECTION,
                    alias_name=ALIAS_NAME,
                )
            )
        ]
    )
    print_json({'alias_name': ALIAS_NAME, 'collection_name': ALIAS_TARGET_COLLECTION})
elif ALIAS_ACTION == 'delete' and CONFIRM_ALIAS_ACTION:
    client.update_collection_aliases(
        change_aliases_operations=[
            qm.DeleteAliasOperation(delete_alias=qm.DeleteAlias(alias_name=ALIAS_NAME))
        ]
    )
    print_json({'deleted_alias': ALIAS_NAME})
else:
    print("Set ALIAS_ACTION to 'point' or 'delete' and CONFIRM_ALIAS_ACTION = True.")

## 17. Delete Collection

Danger zone. This cell previews the collection first and only deletes when `CONFIRM_DELETE_COLLECTION = True`.

In [ ]:
DELETE_COLLECTION = None
CONFIRM_DELETE_COLLECTION = False

if DELETE_COLLECTION:
    print_json(
        {
            'collection': DELETE_COLLECTION,
            'summary': collection_summary(DELETE_COLLECTION),
            'aliases_pointing_here': [
                row for row in alias_rows() if row['collection_name'] == DELETE_COLLECTION
            ],
            'attestation_payload': vector_store(DELETE_COLLECTION).read_meta(),
        }
    )

if DELETE_COLLECTION and CONFIRM_DELETE_COLLECTION:
    aliases_pointing_here = [row for row in alias_rows() if row['collection_name'] == DELETE_COLLECTION]
    if aliases_pointing_here:
        raise RuntimeError(f'Refusing to delete aliased collection: {aliases_pointing_here}')
    client.delete_collection(collection_name=DELETE_COLLECTION)
    print(f'Deleted collection: {DELETE_COLLECTION}')
else:
    print('Set DELETE_COLLECTION and CONFIRM_DELETE_COLLECTION = True to delete an unaliased collection.')

## 18. Quick Operator Checklist

Before a manual alias promotion:
1. Inspect the target collection summary.
2. Read the attestation and confirm `kb_id`, `collection_name`, `retrieval_capability`, and `chunk_count`.
3. Create a snapshot if the operation is risky.
4. Prefer the lifecycle CLI promotion command.
5. Re-run alias health check after the change.

For retrieval quality diagnosis, use a separate retrieval smoke notebook or service-level tests. This notebook answers: "what is Qdrant actually storing and pointing at?"